# LeetCode 2937 — Make Three Strings Equal bằng Python

Link đề: https://leetcode.com/problems/make-three-strings-equal/ · Độ khó: **Easy** · Chủ đề: String, Prefix

Skeleton LeetCode:

```python
class Solution:
    def findMinimumOperations(self, s1: str, s2: str, s3: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả bằng một lời giải duyệt rộng độc lập), nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có ba chuỗi; mỗi thao tác **xóa ký tự cuối cùng** của một chuỗi có độ dài **ít nhất 2**; tìm **số thao tác nhỏ nhất** để ba chuỗi **bằng nhau**, hoặc `-1` nếu không thể.
2. Nhận ra **kết quả cuối cùng phải là một tiền tố chung** của cả ba chuỗi, và muốn ít thao tác nhất thì chọn **tiền tố chung dài nhất** (LCP, longest common prefix).
3. Đi từ cách **mô phỏng thao tác (cắt chuỗi dài nhất cho tới khi bằng nhau)** → **thử từng độ dài đích** → **tính LCP bằng chỉ số** → **`zip` ba chuỗi** → **`os.path.commonprefix`**.
4. Biết **công thức** `tổng độ dài − 3 × LCP` và **điều kiện `-1`** (LCP bằng `0`, vì không thể xóa chuỗi thành rỗng).
5. Tránh các lỗi kinh điển: quên điều kiện `-1`, tính LCP chỉ của hai chuỗi, vượt quá chuỗi ngắn nhất khi duyệt chỉ số, và hiểu nhầm thao tác (xóa ký tự **cuối**, không phải ký tự bất kỳ).

## 2. Đề bài & ví dụ

Cho ba chuỗi `s1`, `s2`, `s3`. Mỗi **thao tác** gồm: chọn **một** trong ba chuỗi có **độ dài ít nhất 2** và **xóa ký tự cuối cùng** của nó.

Hãy trả về **số thao tác nhỏ nhất** để làm cho **ba chuỗi bằng nhau**. Nếu **không thể**, trả về `-1`.

**Ví dụ 1**

- Input: `s1 = "abc"`, `s2 = "abb"`, `s3 = "ab"`
- Output: `2`
- Giải thích: thực hiện một thao tác trên `s1` (`"abc" -> "ab"`) và một thao tác trên `s2` (`"abb" -> "ab"`); khi đó cả ba đều là `"ab"`. Không thể làm ít hơn 2 thao tác.

**Ví dụ 2**

- Input: `s1 = "dac"`, `s2 = "bac"`, `s3 = "cac"`
- Output: `-1`
- Giải thích: ký tự đầu của ba chuỗi khác nhau (`d`, `b`, `c`). Các thao tác chỉ xóa ký tự cuối (và không thể xóa tới khi rỗng), nên ký tự đầu luôn còn nguyên, ba chuỗi không bao giờ bằng nhau.

**Ràng buộc:** `1 <= s1.length, s2.length, s3.length <= 100`; `s1`, `s2`, `s3` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
import os
import random
from collections import deque

examples = [
    ("abc", "abb", "ab", 2),
    ("dac", "bac", "cac", -1),
]
for s1, s2, s3, expected in examples:
    print(f"s1={s1!r}, s2={s2!r}, s3={s3!r} -> kỳ vọng {expected}")

s1='abc', s2='abb', s3='ab' -> kỳ vọng 2
s1='dac', s2='bac', s3='cac' -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Chuỗi, độ dài, chỉ số và cắt lát

Chuỗi là dãy ký tự. `len(s)` là độ dài; `s[i]` là ký tự thứ `i` (từ `0`); `s[:k]` là **tiền tố** gồm `k` ký tự đầu; `s[:-1]` là chuỗi **bỏ ký tự cuối** (chính là một thao tác của đề). Chuỗi trong Python **không đổi được tại chỗ**, nên "xóa ký tự cuối" nghĩa là tạo chuỗi mới `s[:-1]`.

In [2]:
s = "abc"
print(len(s), s[0], s[-1])        # 3 a c
print(s[:2], s[:-1])              # ab ab: tiền tố 2 ký tự và "bỏ ký tự cuối"
print(s[:0] == "")                # True: tiền tố rỗng

3 a c
ab ab
True


### 3.2. Tiền tố chung và tiền tố chung dài nhất (LCP)

Chuỗi `p` là **tiền tố** của `s` nếu `s` bắt đầu bằng `p`. **Tiền tố chung** của ba chuỗi là chuỗi là tiền tố của cả ba. **LCP** là tiền tố chung **dài nhất**. Ví dụ ba chuỗi `"abc"`, `"abb"`, `"ab"` có tiền tố chung `""`, `"a"`, `"ab"`; dài nhất là `"ab"` (`LCP = 2`).

In [3]:
a, b, c = "abc", "abb", "ab"
for k in range(0, 4):
    ok = a[:k] == b[:k] == c[:k] and k <= min(len(a), len(b), len(c))
    print(f"k={k}: {a[:k]!r}, {b[:k]!r}, {c[:k]!r} -> {'là tiền tố chung' if ok else 'không'}")

k=0: '', '', '' -> là tiền tố chung
k=1: 'a', 'a', 'a' -> là tiền tố chung
k=2: 'ab', 'ab', 'ab' -> là tiền tố chung
k=3: 'abc', 'abb', 'ab' -> không


### 3.3. Thao tác chỉ xóa ký tự cuối => kết quả là một tiền tố

Mỗi thao tác bỏ ký tự cuối, nên sau một số thao tác, chuỗi còn lại luôn là **một tiền tố** của chuỗi ban đầu (độ dài giảm dần). Để ba chuỗi bằng nhau, chuỗi cuối cùng `T` phải là **tiền tố của cả ba chuỗi gốc** (tiền tố chung). Hơn nữa, một thao tác chỉ làm được khi độ dài **>= 2**, nên `T` **không thể rỗng**: `|T| >= 1`.

In [4]:
s = "abc"
t = s
for step in range(2):
    t = t[:-1]                       # một thao tác
    print(step + 1, repr(t), "là tiền tố của", repr(s), ":", s.startswith(t))
print("không thể thao tác nữa vì độ dài", len(t), "< 2")

1 'ab' là tiền tố của 'abc' : True
2 'a' là tiền tố của 'abc' : True
không thể thao tác nữa vì độ dài 1 < 2


### 3.4. Số thao tác để đưa một chuỗi về độ dài `L`

Để đưa chuỗi `s` về một tiền tố độ dài `L` cần **`len(s) − L`** thao tác (mỗi thao tác bớt một ký tự). Với ba chuỗi cùng đưa về độ dài `L`: tổng số thao tác là `(|s1| − L) + (|s2| − L) + (|s3| − L) = |s1| + |s2| + |s3| − 3L`. Hàm này **giảm khi `L` tăng**, nên muốn ít thao tác nhất ta chọn `L` **lớn nhất** có thể, tức `L = LCP`.

In [5]:
s1, s2, s3 = "abc", "abb", "ab"
total_len = len(s1) + len(s2) + len(s3)
for L in range(0, 3):
    print(f"L={L}: số thao tác = {total_len} - 3*{L} = {total_len - 3 * L}")

L=0: số thao tác = 8 - 3*0 = 8
L=1: số thao tác = 8 - 3*1 = 5
L=2: số thao tác = 8 - 3*2 = 2


### 3.5. Tính LCP bằng chỉ số

Duyệt `i = 0, 1, 2, ...` và dừng khi: (1) `i` chạm hết một trong ba chuỗi, hoặc (2) ba ký tự `s1[i], s2[i], s3[i]` **không cùng nhau**. Số `i` đã qua chính là LCP. Phải kiểm tra `i < min(len(...))` **trước** khi đọc `s[i]` để tránh `IndexError`.

In [6]:
def lcp_index(a, b, c):
    i = 0
    limit = min(len(a), len(b), len(c))
    while i < limit and a[i] == b[i] == c[i]:
        i += 1
    return i


print(lcp_index("abc", "abb", "ab"))       # 2
print(lcp_index("dac", "bac", "cac"))      # 0
print(lcp_index("same", "same", "same"))   # 4

2
0
4


### 3.6. `zip` ghép các chuỗi theo từng vị trí

`zip(s1, s2, s3)` ghép ký tự cùng vị trí thành bộ ba `(c1, c2, c3)` và **dừng khi chuỗi ngắn nhất hết**: đúng điều ta cần để không vượt quá chuỗi ngắn nhất. Kiểm tra "ba ký tự bằng nhau" bằng `c1 == c2 == c3` hoặc `len({c1, c2, c3}) == 1`.

In [7]:
for triple in zip("abc", "abb", "ab"):
    print(triple, "cùng nhau" if len(set(triple)) == 1 else "khác nhau")

('a', 'a', 'a') cùng nhau
('b', 'b', 'b') cùng nhau


### 3.7. `os.path.commonprefix`

Thư viện chuẩn có hàm `os.path.commonprefix(list)` trả về **tiền tố chung dài nhất** của một danh sách chuỗi (tính từng **ký tự**, không phải từng thành phần đường dẫn). Dùng cho bài này rất gọn: `len(os.path.commonprefix([s1, s2, s3]))`.

In [8]:
print(repr(os.path.commonprefix(["abc", "abb", "ab"])))     # 'ab'
print(repr(os.path.commonprefix(["dac", "bac", "cac"])))    # ''
print(len(os.path.commonprefix(["interview", "internet", "interval"])))   # 5 ('inter')

'ab'
''
5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy xếp ba chuỗi **thẳng hàng theo đầu trái** và nhìn từng cột:

```
s1 = a b c
s2 = a b b
s3 = a b
cột:  1 2 3
```

- cột 1: `a, a, a`: giống nhau;
- cột 2: `b, b, b`: giống nhau;
- cột 3: `c, b, (hết)`: chuỗi `s3` đã hết, và `c != b`.

Phần chung là `"ab"` (LCP = 2). Muốn ba chuỗi bằng nhau, ta cắt đuôi mỗi chuỗi để còn `"ab"`: `s1` bớt 1, `s2` bớt 1, `s3` bớt 0, tổng `2` thao tác.

Nếu **cột 1** đã khác nhau (ví dụ `d, b, c`) thì không có tiền tố chung nào khác rỗng, mà ta **không thể xóa tới rỗng** (cần độ dài `>= 2` mới được xóa), nên đáp án `-1`.

## 5. Cách 1 — Brute Force / baseline: mô phỏng thao tác

**Ý tưởng.** Làm đúng như đề cho phép, theo chiến lược hiển nhiên: **trong khi ba chuỗi chưa bằng nhau**, cắt ký tự cuối của **những chuỗi dài nhất** (một chuỗi dài hơn mọi chuỗi khác chắc chắn phải bị cắt). Nếu mọi chuỗi đều chỉ còn một ký tự mà vẫn khác nhau thì `-1` (không thể thao tác thêm).

In [9]:
def operations_simulate(s1: str, s2: str, s3: str) -> int:
    strings = [s1, s2, s3]
    ops = 0
    while not (strings[0] == strings[1] == strings[2]):
        longest = max(len(s) for s in strings)
        if longest == 1:                           # không thể thao tác nữa (độ dài < 2)
            return -1
        for i in range(3):
            if len(strings[i]) == longest:         # cắt mọi chuỗi đang dài nhất
                strings[i] = strings[i][:-1]       # một thao tác
                ops += 1
    return ops


for s1, s2, s3, expected in examples:
    print(repr(s1), repr(s2), repr(s3), "->", operations_simulate(s1, s2, s3), "(kỳ vọng", expected, ")")

'abc' 'abb' 'ab' -> 2 (kỳ vọng 2 )
'dac' 'bac' 'cac' -> -1 (kỳ vọng -1 )


In [10]:
# Quan sát từng thao tác của ví dụ 1
strings = ["abc", "abb", "ab"]
ops = 0
print("bắt đầu:", strings)
while not (strings[0] == strings[1] == strings[2]):
    longest = max(len(s) for s in strings)
    for i in range(3):
        if len(strings[i]) == longest:
            strings[i] = strings[i][:-1]
            ops += 1
            print(f"cắt chuỗi {i + 1} -> {strings} (ops={ops})")
print("kết thúc:", strings, "| ops =", ops)

bắt đầu: ['abc', 'abb', 'ab']
cắt chuỗi 1 -> ['ab', 'abb', 'ab'] (ops=1)
cắt chuỗi 2 -> ['ab', 'ab', 'ab'] (ops=2)
kết thúc: ['ab', 'ab', 'ab'] | ops = 2


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài lớn nhất của các chuỗi. Có tối đa `3n` thao tác; mỗi thao tác tạo chuỗi mới `O(n)` và so sánh `O(n)`: **Time `O(n²)`**; ba chuỗi mới: **Space `O(n)`**.

**Nút thắt.** Việc tạo chuỗi mới và so sánh lại sau mỗi thao tác là thừa. Kết quả cuối cùng chỉ phụ thuộc vào **độ dài của tiền tố chung dài nhất**, tính được bằng một lượt duyệt.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát.** Chuỗi kết quả `T` là tiền tố chung của cả ba và `|T| >= 1`. Số thao tác là `|s1| + |s2| + |s3| − 3|T|`, nên chọn `|T|` **lớn nhất** = **LCP**.

- Nếu **`LCP = 0`** (ký tự đầu khác nhau, hoặc không tồn tại tiền tố chung không rỗng): không thể (vì `|T| >= 1`) → **`-1`**.
- Ngược lại đáp án là **`len(s1) + len(s2) + len(s3) − 3·LCP`**.

Việc còn lại chỉ là **tính LCP** của ba chuỗi: bằng chỉ số (Cách 3), `zip` (Cách 4), hoặc thư viện (Cách 5). Cách 2 kiểm tra từng độ dài đích để thấy lý do vì sao chọn dài nhất.

## 7. Các cách tiếp theo

### Cách 2 — Thử từng độ dài đích `L` từ dài xuống ngắn

**Giải thích.** Với `L` từ `min(len)` xuống `1`: nếu ba tiền tố độ dài `L` bằng nhau (`s1[:L] == s2[:L] == s3[:L]`) thì trả `tổng − 3L` (độ dài đầu tiên thỏa là lớn nhất, nên ít thao tác nhất). Nếu không `L` nào thỏa, trả `-1`.

In [11]:
def operations_try_lengths(s1: str, s2: str, s3: str) -> int:
    total = len(s1) + len(s2) + len(s3)
    for L in range(min(len(s1), len(s2), len(s3)), 0, -1):    # L = min ... 1 (không gồm 0)
        if s1[:L] == s2[:L] == s3[:L]:                          # ba tiền tố độ dài L giống nhau
            return total - 3 * L
    return -1                                                   # không có tiền tố chung không rỗng


for s1, s2, s3, expected in examples:
    print(repr(s1), repr(s2), repr(s3), "->", operations_try_lengths(s1, s2, s3), "(kỳ vọng", expected, ")")

'abc' 'abb' 'ab' -> 2 (kỳ vọng 2 )
'dac' 'bac' 'cac' -> -1 (kỳ vọng -1 )


In [12]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def findMinimumOperations(self, s1: str, s2: str, s3: str) -> int:
        total = len(s1) + len(s2) + len(s3)
        for L in range(min(len(s1), len(s2), len(s3)), 0, -1):
            if s1[:L] == s2[:L] == s3[:L]:
                return total - 3 * L
        return -1

**Độ phức tạp Cách 2:** **Time `O(n²)`** (tối đa `n` giá trị `L`, mỗi lần so sánh `O(n)`), **Space `O(n)`** (các lát cắt).

### Cách 3 — Tính LCP bằng chỉ số `i`

**Giải thích.** Cho `i` chạy từ `0` khi còn `i < min(len)` và `s1[i] == s2[i] == s3[i]`. Số bước `i` là LCP. Nếu `i == 0` trả `-1`, ngược lại trả `tổng − 3i`.

In [13]:
def operations_lcp_index(s1: str, s2: str, s3: str) -> int:
    limit = min(len(s1), len(s2), len(s3))
    i = 0
    while i < limit and s1[i] == s2[i] == s3[i]:               # kiểm tra giới hạn trước khi đọc s[i]
        i += 1
    if i == 0:
        return -1
    return len(s1) + len(s2) + len(s3) - 3 * i


for s1, s2, s3, expected in examples:
    print(repr(s1), repr(s2), repr(s3), "->", operations_lcp_index(s1, s2, s3), "(kỳ vọng", expected, ")")

'abc' 'abb' 'ab' -> 2 (kỳ vọng 2 )
'dac' 'bac' 'cac' -> -1 (kỳ vọng -1 )


In [14]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def findMinimumOperations(self, s1: str, s2: str, s3: str) -> int:
        limit = min(len(s1), len(s2), len(s3))
        i = 0
        while i < limit and s1[i] == s2[i] == s3[i]:
            i += 1
        return -1 if i == 0 else len(s1) + len(s2) + len(s3) - 3 * i

**Độ phức tạp Cách 3:** **Time `O(n)`** (một lượt duyệt tới LCP), **Space `O(1)`**.

### Cách 4 — `zip` ba chuỗi

**Giải thích.** `zip(s1, s2, s3)` ghép ký tự cùng vị trí và tự dừng ở chuỗi ngắn nhất. Đếm số bộ ba **liên tiếp giống nhau từ đầu**: dừng ngay khi gặp bộ khác nhau.

In [15]:
def operations_zip(s1: str, s2: str, s3: str) -> int:
    lcp = 0
    for a, b, c in zip(s1, s2, s3):
        if a == b == c:
            lcp += 1
        else:
            break                                      # gặp cột khác nhau: dừng
    return -1 if lcp == 0 else len(s1) + len(s2) + len(s3) - 3 * lcp


for s1, s2, s3, expected in examples:
    print(repr(s1), repr(s2), repr(s3), "->", operations_zip(s1, s2, s3), "(kỳ vọng", expected, ")")

'abc' 'abb' 'ab' -> 2 (kỳ vọng 2 )
'dac' 'bac' 'cac' -> -1 (kỳ vọng -1 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 5 — `os.path.commonprefix`

**Giải thích.** Thư viện tính sẵn tiền tố chung dài nhất. `lcp = len(os.path.commonprefix([s1, s2, s3]))`.

In [16]:
def operations_commonprefix(s1: str, s2: str, s3: str) -> int:
    lcp = len(os.path.commonprefix([s1, s2, s3]))
    return -1 if lcp == 0 else len(s1) + len(s2) + len(s3) - 3 * lcp


for s1, s2, s3, expected in examples:
    print(repr(s1), repr(s2), repr(s3), "->", operations_commonprefix(s1, s2, s3), "(kỳ vọng", expected, ")")

'abc' 'abb' 'ab' -> 2 (kỳ vọng 2 )
'dac' 'bac' 'cac' -> -1 (kỳ vọng -1 )


In [17]:
# Code nộp LeetCode cho Cách 5
import os

class Solution:
    def findMinimumOperations(self, s1: str, s2: str, s3: str) -> int:
        lcp = len(os.path.commonprefix([s1, s2, s3]))
        return -1 if lcp == 0 else len(s1) + len(s2) + len(s3) - 3 * lcp

**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

> Cách 3 (LCP bằng chỉ số) là cách nộp rõ ràng nhất; Cách 4 và 5 ngắn hơn. Cách 1 và 2 giúp thấy **vì sao** đáp án là `tổng − 3·LCP`.

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3 tìm LCP cho `("abc", "abb", "ab")` và cho một ví dụ có LCP dài hơn; (b) Cách 2 thử từng độ dài đích.

In [18]:
def trace_lcp(s1, s2, s3):
    limit = min(len(s1), len(s2), len(s3))
    print(f"s1={s1!r} s2={s2!r} s3={s3!r} | limit = min độ dài = {limit}")
    i = 0
    while i < limit:
        triple = (s1[i], s2[i], s3[i])
        same = triple[0] == triple[1] == triple[2]
        print(f"  i={i}: ký tự {triple} -> {'giống nhau, LCP tăng' if same else 'khác nhau, dừng'}")
        if not same:
            break
        i += 1
    else:
        print(f"  i={i} = limit: hết chuỗi ngắn nhất, dừng")
    total = len(s1) + len(s2) + len(s3)
    answer = -1 if i == 0 else total - 3 * i
    print(f"  => LCP = {i}, tổng độ dài = {total}, đáp án = {answer if i else '-1 (LCP = 0)'}")
    return answer


assert trace_lcp("abc", "abb", "ab") == 2
print()
assert trace_lcp("leet", "leetcode", "leetc") == 4 + 8 + 5 - 3 * 4
print()
assert trace_lcp("dac", "bac", "cac") == -1

s1='abc' s2='abb' s3='ab' | limit = min độ dài = 2
  i=0: ký tự ('a', 'a', 'a') -> giống nhau, LCP tăng
  i=1: ký tự ('b', 'b', 'b') -> giống nhau, LCP tăng
  i=2 = limit: hết chuỗi ngắn nhất, dừng
  => LCP = 2, tổng độ dài = 8, đáp án = 2

s1='leet' s2='leetcode' s3='leetc' | limit = min độ dài = 4
  i=0: ký tự ('l', 'l', 'l') -> giống nhau, LCP tăng
  i=1: ký tự ('e', 'e', 'e') -> giống nhau, LCP tăng
  i=2: ký tự ('e', 'e', 'e') -> giống nhau, LCP tăng
  i=3: ký tự ('t', 't', 't') -> giống nhau, LCP tăng
  i=4 = limit: hết chuỗi ngắn nhất, dừng
  => LCP = 4, tổng độ dài = 17, đáp án = 5

s1='dac' s2='bac' s3='cac' | limit = min độ dài = 3
  i=0: ký tự ('d', 'b', 'c') -> khác nhau, dừng
  => LCP = 0, tổng độ dài = 9, đáp án = -1 (LCP = 0)


In [19]:
def trace_try_lengths(s1, s2, s3):
    total = len(s1) + len(s2) + len(s3)
    print(f"tổng độ dài = {total}")
    for L in range(min(len(s1), len(s2), len(s3)), 0, -1):
        prefixes = (s1[:L], s2[:L], s3[:L])
        ok = prefixes[0] == prefixes[1] == prefixes[2]
        print(f"  L={L}: tiền tố {prefixes} -> {'bằng nhau: trả ' + str(total - 3 * L) if ok else 'khác nhau'}")
        if ok:
            return total - 3 * L
    print("  không có L nào -> -1")
    return -1


assert trace_try_lengths("abc", "abb", "ab") == 2

tổng độ dài = 8
  L=2: tiền tố ('ab', 'ab', 'ab') -> bằng nhau: trả 2


Bảng tóm tắt (Cách 3, `("abc", "abb", "ab")`):

| `i` | `(s1[i], s2[i], s3[i])` | kết quả |
|---|---|---|
| 0 | `(a, a, a)` | giống nhau, `i = 1` |
| 1 | `(b, b, b)` | giống nhau, `i = 2` |
| 2 | không xét (`limit = 2`) | dừng, `LCP = 2` |

Đáp án `3 + 3 + 2 − 3·2 = 2`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (LCP bằng chỉ số). Đúng class `Solution`, đúng method `findMinimumOperations(self, s1: str, s2: str, s3: str) -> int`.

In [20]:
class Solution:
    def findMinimumOperations(self, s1: str, s2: str, s3: str) -> int:
        limit = min(len(s1), len(s2), len(s3))     # không thể dài hơn chuỗi ngắn nhất
        i = 0
        while i < limit and s1[i] == s2[i] == s3[i]:
            i += 1                                  # i = độ dài tiền tố chung
        if i == 0:                                  # không có tiền tố chung không rỗng
            return -1
        return len(s1) + len(s2) + len(s3) - 3 * i


sol = Solution()
for s1, s2, s3, expected in examples:
    print(repr(s1), repr(s2), repr(s3), "->", sol.findMinimumOperations(s1, s2, s3))

'abc' 'abb' 'ab' -> 2
'dac' 'bac' 'cac' -> -1


## 10. Vì sao thuật toán đúng

- **Chuỗi đích là tiền tố chung:** mỗi thao tác bỏ ký tự cuối, nên mọi chuỗi luôn là tiền tố của chuỗi gốc của nó. Để ba chuỗi bằng nhau, chuỗi chung `T` phải là tiền tố của cả ba (tiền tố chung).
- **Không thể rỗng:** thao tác chỉ làm được khi độ dài `>= 2`, nên mỗi chuỗi sau mọi thao tác có độ dài `>= 1`; vậy `|T| >= 1`.
- **Chi phí theo `|T| = L`:** mỗi chuỗi cần đúng `len − L` thao tác (không thể ít hơn vì cần bỏ `len − L` ký tự, và đủ vì mỗi thao tác bỏ một ký tự), tổng `tổng − 3L`.
- **Chọn `L` lớn nhất:** chi phí giảm khi `L` tăng, nên chọn `L` lớn nhất sao cho `T` là tiền tố chung, tức `L = LCP`. Nếu `LCP = 0` thì không có `T` hợp lệ: `-1`.
- **Tính chất kiểm được:** nếu đáp án `k >= 0` thì `k + 3·LCP = tổng độ dài`; đáp án chỉ `-1` khi các ký tự đầu của ba chuỗi **không cùng nhau**.

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài lớn nhất của ba chuỗi.

- **Cách 1 (mô phỏng):** Time `O(n²)`, Space `O(n)`.
- **Cách 2 (thử từng `L`):** Time `O(n²)`, Space `O(n)`.
- **Cách 3, 4, 5 (LCP):** Time **`O(n)`**, Space **`O(1)`**.
- Mọi cách phải đọc ít nhất tới điểm khác nhau đầu tiên, nên không thể tốt hơn `O(LCP)`; ở trường hợp xấu nhất `O(n)`.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`. Ngoài ra, ta viết một **lời giải duyệt rộng (BFS) độc lập** để làm "đáp án thật" cho các chuỗi ngắn.

In [21]:
sol = Solution()
impls = {
    "simulate": operations_simulate,
    "try_lengths": operations_try_lengths,
    "lcp_index": operations_lcp_index,
    "zip": operations_zip,
    "commonprefix": operations_commonprefix,
    "submit": sol.findMinimumOperations,
}


def bfs_oracle(s1, s2, s3):
    """Đáp án thật bằng BFS trên trạng thái (a, b, c): mỗi bước xóa ký tự cuối của một chuỗi dài >= 2."""
    start = (s1, s2, s3)
    seen = {start: 0}
    queue = deque([start])
    while queue:
        state = queue.popleft()
        if state[0] == state[1] == state[2]:
            return seen[state]
        for i in range(3):
            if len(state[i]) >= 2:
                nxt = list(state)
                nxt[i] = nxt[i][:-1]
                nxt = tuple(nxt)
                if nxt not in seen:
                    seen[nxt] = seen[state] + 1
                    queue.append(nxt)
    return -1


for s1, s2, s3, expected in examples:
    assert bfs_oracle(s1, s2, s3) == expected
    for key, fn in impls.items():
        assert fn(s1, s2, s3) == expected, (key, s1, s2, s3, fn(s1, s2, s3), expected)
print("PASS ví dụ của đề (mọi cách và BFS độc lập)")

PASS ví dụ của đề (mọi cách và BFS độc lập)


### 12.1. Các trường hợp biên

In [22]:
def check_all(s1, s2, s3, expected, label):
    assert bfs_oracle(s1, s2, s3) == expected, (label, "oracle", bfs_oracle(s1, s2, s3), expected)
    for key, fn in impls.items():
        got = fn(s1, s2, s3)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all("a", "a", "a", 0, "ba chuỗi một ký tự giống nhau")
check_all("a", "b", "a", -1, "ba chuỗi một ký tự không giống nhau")
check_all("abc", "abc", "abc", 0, "đã bằng nhau")
check_all("abc", "ab", "a", 3, "cùng tiền tố 'a': 2 + 1 + 0 = 3")
check_all("ab", "ab", "abc", 1, "chỉ cần cắt một chuỗi")
check_all("abc", "bac", "cab", -1, "ký tự đầu khác nhau")
check_all("abc", "abd", "abe", 3, "LCP = 2, mỗi chuỗi cắt 1")
check_all("aaa", "aa", "a", 3, "toàn chữ a, LCP = 1: 2 + 1 + 0")
check_all("a" * 100, "a" * 99, "a" * 98, 3, "chuỗi dài nhất của đề")

PASS ba chuỗi một ký tự giống nhau -> 0
PASS ba chuỗi một ký tự không giống nhau -> -1
PASS đã bằng nhau -> 0
PASS cùng tiền tố 'a': 2 + 1 + 0 = 3 -> 3
PASS chỉ cần cắt một chuỗi -> 1
PASS ký tự đầu khác nhau -> -1
PASS LCP = 2, mỗi chuỗi cắt 1 -> 3
PASS toàn chữ a, LCP = 1: 2 + 1 + 0 -> 3
PASS chuỗi dài nhất của đề -> 3


### 12.2. Tính chất và phản ví dụ

In [23]:
# Tính chất 1: tổng độ dài = đáp án + 3 * LCP khi đáp án != -1
random.seed(29)
for _ in range(300):
    a, b, c = ("".join(random.choice("ab") for _ in range(random.randint(1, 12))) for _ in range(3))
    r = sol.findMinimumOperations(a, b, c)
    lcp = len(os.path.commonprefix([a, b, c]))
    if r == -1:
        assert lcp == 0 and not (a[0] == b[0] == c[0])
    else:
        assert r + 3 * lcp == len(a) + len(b) + len(c) and lcp >= 1 and a[0] == b[0] == c[0]
print("PASS đáp án + 3·LCP = tổng độ dài; -1 khi ký tự đầu không cùng nhau")

# Tính chất 2: đối xứng khi hoán vị ba chuỗi
for _ in range(200):
    a, b, c = ("".join(random.choice("abc") for _ in range(random.randint(1, 10))) for _ in range(3))
    assert sol.findMinimumOperations(a, b, c) == sol.findMinimumOperations(c, a, b) == sol.findMinimumOperations(b, c, a)
print("PASS hoán vị ba chuỗi không đổi đáp án")

# Đáp án = 0 khi và chỉ khi ba chuỗi bằng nhau
assert sol.findMinimumOperations("xyz", "xyz", "xyz") == 0
assert sol.findMinimumOperations("xyz", "xy", "xyz") == 2          # cắt hai chuỗi "xyz"
print("PASS đáp án 0 khi ba chuỗi bằng nhau")

# Phản ví dụ 1: quên điều kiện -1
def sai_quen_tru_1(s1, s2, s3):
    lcp = len(os.path.commonprefix([s1, s2, s3]))
    return len(s1) + len(s2) + len(s3) - 3 * lcp          # khi lcp = 0 trả tổng độ dài

assert sai_quen_tru_1("dac", "bac", "cac") == 9 and sol.findMinimumOperations("dac", "bac", "cac") == -1
print("PASS phản ví dụ: quên -1 cho", sai_quen_tru_1("dac", "bac", "cac"), "thay vì -1")

# Phản ví dụ 2: chỉ tính LCP của hai chuỗi đầu
def sai_hai_chuoi(s1, s2, s3):
    lcp = len(os.path.commonprefix([s1, s2]))
    return -1 if lcp == 0 else len(s1) + len(s2) + len(s3) - 3 * lcp

assert sai_hai_chuoi("abc", "abb", "x") != sol.findMinimumOperations("abc", "abb", "x")
print("PASS phản ví dụ: bỏ qua s3 cho", sai_hai_chuoi("abc", "abb", "x"), "thay vì -1")

PASS đáp án + 3·LCP = tổng độ dài; -1 khi ký tự đầu không cùng nhau
PASS hoán vị ba chuỗi không đổi đáp án
PASS đáp án 0 khi ba chuỗi bằng nhau
PASS phản ví dụ: quên -1 cho 9 thay vì -1
PASS phản ví dụ: bỏ qua s3 cho 1 thay vì -1


### 12.3. Đối chiếu ngẫu nhiên (có seed) và với BFS độc lập

In [24]:
random.seed(2937)
# (1) Chuỗi ngắn: đối chiếu mọi cách với BFS độc lập
for trial in range(1500):
    a, b, c = ("".join(random.choice("ab") for _ in range(random.randint(1, 6))) for _ in range(3))
    truth = bfs_oracle(a, b, c)
    for key, fn in impls.items():
        assert fn(a, b, c) == truth, (key, a, b, c, fn(a, b, c), truth)
print("PASS 1500 bộ chuỗi ngắn: mọi cách khớp BFS độc lập")

# (2) Chuỗi dài (tới 100 ký tự): đối chiếu các cách với nhau (BFS quá chậm)
for trial in range(1000):
    alphabet = random.choice(["a", "ab", "abc"])
    common = "".join(random.choice(alphabet) for _ in range(random.randint(0, 50)))
    a, b, c = (common + "".join(random.choice(alphabet) for _ in range(random.randint(0, 50))) for _ in range(3))
    if not a or not b or not c:
        continue
    ref = operations_simulate(a, b, c)
    for key, fn in impls.items():
        assert fn(a, b, c) == ref, (key, a, b, c, fn(a, b, c), ref)
print("PASS 1000 bộ chuỗi dài (tới 100) khớp nhau")

PASS 1500 bộ chuỗi ngắn: mọi cách khớp BFS độc lập


PASS 1000 bộ chuỗi dài (tới 100) khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `-1`

In [25]:
def sai_quen_tru_1(s1, s2, s3):
    lcp = len(os.path.commonprefix([s1, s2, s3]))
    return len(s1) + len(s2) + len(s3) - 3 * lcp


print("sai  :", sai_quen_tru_1("dac", "bac", "cac"), "| đúng:", Solution().findMinimumOperations("dac", "bac", "cac"))

sai  : 9 | đúng: -1


**Vì sao sai:** khi `LCP = 0` công thức cho `9` (tổng độ dài), tức "xóa hết mọi ký tự"; nhưng thao tác chỉ làm được khi độ dài `>= 2` nên **không thể** xóa tới rỗng. Phải trả `-1` khi `LCP = 0`.

### Lỗi 2: Chỉ tính LCP của hai chuỗi

In [26]:
def sai_hai_chuoi(s1, s2, s3):
    lcp = len(os.path.commonprefix([s1, s2]))                  # bỏ qua s3
    return -1 if lcp == 0 else len(s1) + len(s2) + len(s3) - 3 * lcp


print("sai  :", sai_hai_chuoi("abc", "abb", "x"), "| đúng:", Solution().findMinimumOperations("abc", "abb", "x"))

sai  : 1 | đúng: -1


**Vì sao sai:** tiền tố chung phải thuộc **cả ba** chuỗi. Bỏ `s3` làm LCP lớn hơn thực tế, trả số thao tác sai (hoặc báo có đáp án trong khi `s3` hoàn toàn không khớp).

### Lỗi 3: Duyệt chỉ số vượt quá chuỗi ngắn nhất

In [27]:
def sai_vuot_gioi_han(s1, s2, s3):
    i = 0
    while s1[i] == s2[i] == s3[i]:                 # không kiểm tra i < độ dài
        i += 1
    return i


try:
    sai_vuot_gioi_han("ab", "ab", "ab")
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().findMinimumOperations("ab", "ab", "ab"))

IndexError: string index out of range
đúng: 0


**Vì sao sai:** khi ba chuỗi giống nhau hoàn toàn, vòng lặp chạy qua hết chiều dài và đọc `s[len]`, gây `IndexError`. Luôn kiểm tra `i < min(len(s1), len(s2), len(s3))` **trước** khi đọc ký tự (hoặc dùng `zip`).

### Lỗi 4: Hiểu nhầm thao tác (xóa ký tự bất kỳ)

In [28]:
def sai_xoa_bat_ky(s1, s2, s3):
    # tưởng được xóa ký tự bất kỳ: so sánh theo tập ký tự thay vì tiền tố
    return 0 if sorted(s1) == sorted(s2) == sorted(s3) else -1


print("sai  :", sai_xoa_bat_ky("abc", "cab", "bca"), "| đúng:", Solution().findMinimumOperations("abc", "cab", "bca"))

sai  : 0 | đúng: -1


**Vì sao sai:** thao tác chỉ xóa **ký tự cuối**, không đổi được thứ tự ký tự. Ba chuỗi `"abc"`, `"cab"`, `"bca"` là hoán vị của nhau nhưng ký tự đầu khác nhau nên không bao giờ bằng nhau: đáp án đúng là `-1`.

### Lỗi 5: Cho phép xóa tới chuỗi rỗng

In [29]:
def sai_cho_phep_rong(s1, s2, s3):
    # coi như mọi chuỗi có thể xóa hết rỗng: ba chuỗi rỗng luôn bằng nhau
    return len(s1) + len(s2) + len(s3) - 3 * len(os.path.commonprefix([s1, s2, s3]))


for a, b, c in [("a", "b", "a"), ("ab", "b", "ab")]:
    print((a, b, c), "sai:", sai_cho_phep_rong(a, b, c), "| đúng:", Solution().findMinimumOperations(a, b, c), "| BFS:", bfs_oracle(a, b, c))

('a', 'b', 'a') sai: 3 | đúng: -1 | BFS: -1
('ab', 'b', 'ab') sai: 5 | đúng: -1 | BFS: -1


**Vì sao sai:** đề chỉ cho xóa khi độ dài **`>= 2`**, nên mỗi chuỗi luôn còn **ít nhất 1 ký tự**; "ba chuỗi rỗng" không bao giờ đạt được. Đây là cùng nguồn gốc với Lỗi 1: kết quả `LCP = 0` ⇒ `-1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "chỉ được xóa/cắt ở **cuối** (hoặc đầu)" → kết quả là một **tiền tố (hậu tố)** của chuỗi gốc.
- Thấy "làm cho nhiều chuỗi bằng nhau" bằng cách chỉ cắt đuôi → nghĩ ngay tới **tiền tố chung dài nhất (LCP)**.
- Công thức: `tổng độ dài − k·LCP` với `k` là số chuỗi; nhớ điều kiện **không rỗng** (`LCP >= 1`).
- Dùng `zip` hoặc kiểm tra `i < min(len)` để không vượt biên khi duyệt song song nhiều chuỗi.
- Có thể kiểm chứng bằng **BFS** trên không gian trạng thái nhỏ (chuỗi ngắn) để chắc công thức đúng.
- Công cụ sẵn có: `os.path.commonprefix`.

## 15. Biến thể của bài

1. Làm cho **hai** chuỗi bằng nhau thay vì ba: `len(s1) + len(s2) − 2·LCP`.
2. **`k` chuỗi** bất kỳ: `Σ len − k·LCP`, với `LCP >= 1`.
3. Cho phép xóa ký tự **đầu** hoặc **cuối**: bài khác (hậu tố/tiền tố hoặc chuỗi con chung dài nhất).
4. Cho phép **xóa tới rỗng**: khi đó `LCP = 0` vẫn hợp lệ, đáp án luôn tồn tại: `Σ len − k·LCP`.
5. Trả về **chuỗi cuối cùng** (tiền tố chung) thay vì số thao tác.
6. Chi phí khác nhau cho mỗi chuỗi (thao tác trên `s_i` tốn `w_i`): tổng `Σ w_i (len_i − LCP)`.

Ta thử biến thể 2, 4 và 5:

In [30]:
def min_ops_k_strings(strings, allow_empty=False):
    lcp = len(os.path.commonprefix(strings))
    if lcp == 0 and not allow_empty:
        return -1
    return sum(len(s) for s in strings) - len(strings) * lcp


def final_string(strings):
    p = os.path.commonprefix(strings)
    return p if p else None


print("2. bốn chuỗi:", min_ops_k_strings(["abcd", "abce", "abx", "abcdef"]))       # 4+4+3+6 - 4*2 = 9
print("4. cho phép rỗng, LCP = 0:", min_ops_k_strings(["dac", "bac", "cac"], allow_empty=True))      # 9
print("5. chuỗi cuối:", final_string(["abc", "abb", "ab"]), final_string(["dac", "bac", "cac"]))
assert min_ops_k_strings(["abc", "abb", "ab"]) == 2

2. bốn chuỗi: 9
4. cho phép rỗng, LCP = 0: 9
5. chuỗi cuối: ab None


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng cắt chuỗi dài nhất | làm đúng theo thao tác | `O(n²)` | `O(n)` | hiểu đề; bản đối chiếu |
| 2. Thử từng độ dài đích `L` | tiền tố bằng nhau, `tổng − 3L` | `O(n²)` | `O(n)` | thấy vì sao chọn `L` lớn nhất |
| 3. LCP bằng chỉ số | `while i < limit and s1[i]==s2[i]==s3[i]` | `O(n)` | `O(1)` | **cách nộp bài** |
| 4. `zip` ba chuỗi | tự dừng ở chuỗi ngắn nhất | `O(n)` | `O(1)` | ngắn gọn, an toàn biên |
| 5. `os.path.commonprefix` | thư viện | `O(n)` | `O(1)` | một dòng; nên biết cách tự viết |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"hello"[:-1]`, `"hello"[:3]`, `len(os.path.commonprefix(["flower", "flow", "flight"]))`, `Solution().findMinimumOperations("ab", "ab", "ab")`, `Solution().findMinimumOperations("b", "a", "a")`.

**Bài 2 (tính tay).** Với `s1 = "leet"`, `s2 = "leetcode"`, `s3 = "leetc"`: tìm LCP, tổng độ dài và đáp án. Làm lại với `s1 = "xyz"`, `s2 = "xy"`, `s3 = "zy"`.

**Bài 3 (điền TODO).** Hoàn thành `operations_todo(s1, s2, s3)` theo Cách 4 (`zip`).

In [31]:
def operations_todo(s1, s2, s3):
    lcp = 0
    # TODO 1: duyệt for a, b, c in zip(s1, s2, s3)
    # TODO 2: nếu a == b == c thì lcp += 1, ngược lại break
    # TODO 3: nếu lcp == 0 trả -1; ngược lại trả tổng độ dài - 3 * lcp
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `operations_simulate2(s1, s2, s3)` bằng cách mô phỏng cắt chuỗi dài nhất (Cách 1) mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_ops_list(strings)` cho **danh sách `k` chuỗi** (trả `-1` nếu `LCP = 0`). (b) Viết `min_ops_weighted(strings, weights)` trong đó thao tác trên chuỗi `i` tốn `weights[i]`; trả tổng chi phí nhỏ nhất hoặc `-1`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'hell'`, `'hel'`, ba từ `flower`, `flow`, `flight` đều bắt đầu bằng `fl` (độ dài `?`), ba chuỗi giống nhau thì `0`, và `"b", "a", "a"` có ký tự đầu khác nhau. Bài 2: LCP là độ dài tiền tố chung dài nhất.
- **Mức 2 (vừa):** Bài 1: `os.path.commonprefix(["flower", "flow", "flight"])` là `'fl'`, nên độ dài `2`; hai đáp án cuối là `0` và `-1`. Bài 2: `leet/leetcode/leetc`: LCP `4`, tổng `4 + 8 + 5 = 17`, đáp án `17 − 12 = 5`; `xyz/xy/zy`: ký tự đầu `x, x, z` không cùng nên `-1`. Bài 3: ba TODO trong comment. Bài 4: xem Cách 1.
- **Mức 3 (sâu):** Bài 5(a): `lcp = len(os.path.commonprefix(strings))`; `-1` nếu `0`, ngược lại `sum(map(len, strings)) − len(strings)·lcp`. Bài 5(b): `Σ weights[i]·(len(strings[i]) − lcp)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert "hello"[:-1] == "hell" and "hello"[:3] == "hel"
assert len(os.path.commonprefix(["flower", "flow", "flight"])) == 2
assert Solution().findMinimumOperations("ab", "ab", "ab") == 0
assert Solution().findMinimumOperations("b", "a", "a") == -1

# Bài 2
assert len(os.path.commonprefix(["leet", "leetcode", "leetc"])) == 4
assert Solution().findMinimumOperations("leet", "leetcode", "leetc") == 17 - 12 == 5
assert Solution().findMinimumOperations("xyz", "xy", "zy") == -1

# Bài 3
def operations_todo_ans(s1, s2, s3):
    lcp = 0
    for a, b, c in zip(s1, s2, s3):
        if a == b == c:
            lcp += 1
        else:
            break
    return -1 if lcp == 0 else len(s1) + len(s2) + len(s3) - 3 * lcp

for s1, s2, s3, expected in examples:
    assert operations_todo_ans(s1, s2, s3) == expected

# Bài 4
def operations_simulate2(s1, s2, s3):
    strings = [s1, s2, s3]
    ops = 0
    while not (strings[0] == strings[1] == strings[2]):
        longest = max(len(s) for s in strings)
        if longest == 1:
            return -1
        for i in range(3):
            if len(strings[i]) == longest:
                strings[i] = strings[i][:-1]
                ops += 1
    return ops

for s1, s2, s3, expected in examples:
    assert operations_simulate2(s1, s2, s3) == expected

# Bài 5(a)
def min_ops_list(strings):
    lcp = len(os.path.commonprefix(strings))
    return -1 if lcp == 0 else sum(map(len, strings)) - len(strings) * lcp

assert min_ops_list(["abc", "abb", "ab"]) == 2
assert min_ops_list(["abcd", "abce", "abx", "abcdef"]) == 9
assert min_ops_list(["a", "b"]) == -1

# Bài 5(b)
def min_ops_weighted(strings, weights):
    lcp = len(os.path.commonprefix(strings))
    if lcp == 0:
        return -1
    return sum(w * (len(s) - lcp) for s, w in zip(strings, weights))

assert min_ops_weighted(["abc", "abb", "ab"], [1, 1, 1]) == 2
assert min_ops_weighted(["abc", "abb", "ab"], [5, 10, 100]) == 15
assert min_ops_weighted(["xa", "ya"], [1, 1]) == -1
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (LCP của ba chuỗi)

```python
class Solution:
    def findMinimumOperations(self, s1: str, s2: str, s3: str) -> int:
        limit = min(len(s1), len(s2), len(s3))
        i = 0
        while i < limit and s1[i] == s2[i] == s3[i]:
            i += 1
        return -1 if i == 0 else len(s1) + len(s2) + len(s3) - 3 * i
```

**Checklist**

- [ ] Hiểu kết quả cuối là **tiền tố chung của cả ba** chuỗi (thao tác chỉ xóa ký tự **cuối**)?
- [ ] Trả `-1` khi `LCP = 0` (không thể xóa tới chuỗi rỗng)?
- [ ] Công thức `len(s1) + len(s2) + len(s3) − 3·LCP` (nhân **3**)?
- [ ] LCP tính trên **cả ba** chuỗi, kiểm tra `i < min(len)` trước khi đọc `s[i]`?
- [ ] Đã thử: ba chuỗi bằng nhau (`0`), ký tự đầu khác nhau (`-1`), một chuỗi là tiền tố của chuỗi khác?
- [ ] Đã đối chiếu với lời giải mô phỏng hoặc BFS trên chuỗi ngắn?
- [ ] Skeleton đúng `class Solution` / `def findMinimumOperations(self, s1: str, s2: str, s3: str) -> int`?